# Minimal Sionna RT City Model

This notebook keeps only the core workflow:

- convert `output_precise.glb` to Sionna/Mitsuba XML + PLY
- load the scene with Sionna RT
- add TX/RX and antenna arrays
- preview the scene with `scene.preview()`
- optionally run ray tracing with `PathSolver()`

Generated files are written under `generated_model/`.


## 1. Configure Model, Radio Nodes, and RT Parameters

In [ ]:
from pathlib import Path
from types import SimpleNamespace
import numpy as np

NOTEBOOK_DIR =  Path("/<your path to>/glb_to_rt") #Path.cwd()

# Input GLB and generated Sionna/Mitsuba files.
MODEL = NOTEBOOK_DIR / "example_chongqing.glb"
CONVERTED_DIR = NOTEBOOK_DIR / "generated_model"
SOURCE_UP = "y"  # glTF/GLB is usually Y-up; Sionna RT uses Z-up.

# Carrier and ray-tracing settings.
FREQUENCY = 3.5e9
BANDWIDTH = 100e6
MAX_DEPTH = 5
REFRACTION = False

# Radio node positions in Sionna coordinates, after the GLB axis conversion.
TX_POSITION = np.array([0.0, 0.0, 25.0], dtype=np.float32)
RX_POSITION = np.array([50.0, 0.0, 1.5], dtype=np.float32)
TX_LOOK_AT = np.array([0.0, 0.0, 0.0], dtype=np.float32)
TX_POWER_DBM = 30.0

# Antenna array settings passed directly to Sionna RT PlanarArray.
TX_ROWS = 4
TX_COLS = 4
RX_ROWS = 1
RX_COLS = 1
TX_PATTERN = "tr38901"
RX_PATTERN = "dipole"
TX_POLARIZATION = "V"
RX_POLARIZATION = "V"

# Sionna RT preview options.
PREVIEW_RESOLUTION = [1280, 720]
PREVIEW_FOV = 45.0
HIDE_DEVICES = True
SHOW_ORIENTATIONS = False

args = SimpleNamespace(
    model=str(MODEL),
    converted_dir=str(CONVERTED_DIR),
    source_up=SOURCE_UP,
    frequency=FREQUENCY,
    bandwidth=BANDWIDTH,
    max_depth=MAX_DEPTH,
    refraction=REFRACTION,
    tx=TX_POSITION,
    rx=RX_POSITION,
    look_at=TX_LOOK_AT,
    tx_power_dbm=TX_POWER_DBM,
    tx_rows=TX_ROWS,
    tx_cols=TX_COLS,
    rx_rows=RX_ROWS,
    rx_cols=RX_COLS,
    tx_pattern=TX_PATTERN,
    rx_pattern=RX_PATTERN,
    tx_polarization=TX_POLARIZATION,
    rx_polarization=RX_POLARIZATION,
    preview_resolution=PREVIEW_RESOLUTION,
    preview_fov=PREVIEW_FOV,
    hide_devices=HIDE_DEVICES,
    show_orientations=SHOW_ORIENTATIONS,
)

print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Model: {MODEL}")


Notebook directory: /root/glb_to_rt
Model: /root/glb_to_rt/example_chongqing.glb


## 2. Define GLB-to-Sionna Format Conversion Functions

These functions are only for format conversion: they read the GLB, assign radio materials, write ASCII PLY meshes, and generate a Sionna/Mitsuba XML scene. They do not add TX/RX, preview the scene, or run ray tracing.

### 2.1 Shared Imports and Supported Sionna Materials

This setup is used by the conversion code. `SUPPORTED_ITU_TYPES` lists the Sionna RT `itu-radio-material` types that the generated XML is allowed to reference.

In [34]:
from collections import Counter
from html import escape
from pathlib import Path
import re
import textwrap
from typing import Any

import numpy as np

from sionna.rt import PathSolver, PlanarArray, Receiver, Transmitter, load_scene

SUPPORTED_ITU_TYPES = {
    "marble",
    "concrete",
    "wood",
    "metal",
    "brick",
    "glass",
    "floorboard",
    "ceiling_board",
    "chipboard",
    "plasterboard",
    "plywood",
    "very_dry_ground",
    "medium_dry_ground",
    "wet_ground",
}


def safe_name(value: str) -> str:
    """Return a filesystem/XML-id-safe name."""
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", value).strip("._") or "city_model"


### 2.2 Material Mapping

These functions define how GLB visual material names or mesh names map to Sionna RT radio materials. The material table is built into the notebook, so no `materials.json` file is needed. Colors are normalized to `[0, 1]` and written into the generated XML for visualization.

In [35]:
def material_config() -> dict[str, Any]:
    """Built-in visual-to-radio material mapping for this city model."""
    return {
        "default_material": "concrete",
        "materials": {
            "asphalt": {"itu_type": "concrete", "thickness": 0.15, "color": [0.439216, 0.458824, 0.478431]},
            "brick": {"itu_type": "brick", "thickness": 0.2, "color": [0.619608, 0.219608, 0.141176]},
            "concrete": {"itu_type": "concrete", "thickness": 0.2, "color": [0.580392, 0.580392, 0.54902]},
            "glass": {"itu_type": "glass", "thickness": 0.05, "color": [0.45098, 0.721569, 0.898039]},
            "grass": {"itu_type": "medium_dry_ground", "thickness": 0.05, "color": [0.2, 0.619608, 0.239216]},
            "forest": {"itu_type": "wood", "thickness": 0.3, "color": [0.05098, 0.360784, 0.121569]},
            "water": {"itu_type": "wet_ground", "thickness": 0.01, "color": [0.019608, 0.321569, 0.898039]},
            "soil": {"itu_type": "medium_dry_ground", "thickness": 0.1, "color": [0.521569, 0.380392, 0.239216]},
            "sand": {"itu_type": "very_dry_ground", "thickness": 0.1, "color": [0.698039, 0.639216, 0.439216]},
        },
        "material_aliases": {
            "road": "asphalt",
            "building_brick": "brick",
            "building_concrete": "concrete",
            "building_glass": "glass",
            "land": "soil",
            "fresh_water": "water",
        },
        "assignments": [
            {"pattern": "building_glass|glass|window|facade", "material": "glass"},
            {"pattern": "building_brick|brick|wall_brick", "material": "brick"},
            {"pattern": "fresh_water|water|river|lake|pool|canal", "material": "water"},
            {"pattern": "road|asphalt|street|highway|sidewalk", "material": "asphalt"},
            {"pattern": "grass|park|meadow|lawn", "material": "grass"},
            {"pattern": "forest|tree|vegetation|woodland", "material": "forest"},
            {"pattern": "soil|land|terrain|floor|ground", "material": "soil"},
            {"pattern": "sand", "material": "sand"},
        ],
    }


def validate_material_config(config: dict[str, Any]) -> dict[str, Any]:
    """Fail early if a configured ITU material is not supported by Sionna RT."""
    materials = config["materials"]
    if config["default_material"] not in materials:
        raise ValueError("Default material is not defined")
    for material_id, material in materials.items():
        itu_type = material.get("itu_type", material_id)
        if itu_type not in SUPPORTED_ITU_TYPES:
            raise ValueError(f"Material {material_id!r} uses unsupported ITU type {itu_type!r}")
    return config


def normalize_rgb(color: Any) -> np.ndarray | None:
    """Normalize RGB colors to [0, 1] before writing them to XML."""
    if color is None:
        return None
    rgb = np.asarray(color, dtype=np.float64).reshape(-1)
    if rgb.size < 3:
        return None
    rgb = rgb[:3]
    if np.nanmax(rgb) > 1.0:
        rgb = rgb / 255.0
    return np.clip(rgb, 0.0, 1.0)


def material_to_xml(material_id: str, material: dict[str, Any]) -> str:
    """Render one Sionna ITU radio material as Mitsuba XML."""
    itu_type = material.get("itu_type", material_id)
    thickness = material.get("thickness", 0.1)
    lines = [
        f'    <bsdf type="itu-radio-material" id="{escape(material_id, quote=True)}">',
        f'        <string name="type" value="{escape(itu_type, quote=True)}"/>',
        f'        <float name="thickness" value="{thickness}"/>',
    ]
    rgb = normalize_rgb(material.get("color"))
    if rgb is not None:
        lines.append(f'        <rgb name="color" value="{rgb[0]:.6g}, {rgb[1]:.6g}, {rgb[2]:.6g}"/>')
    lines.append("    </bsdf>")
    return "\n".join(lines)


def mesh_material_name(mesh: Any) -> str | None:
    """Read the GLB visual material name from a mesh, if present."""
    material = getattr(getattr(mesh, "visual", None), "material", None)
    name = getattr(material, "name", None)
    return safe_name(str(name)) if name else None


def material_for_shape(shape_name: str, config: dict[str, Any], source_material: str | None = None) -> str:
    """Choose a radio material from GLB material names and mesh names."""
    materials = config["materials"]
    aliases = config.get("material_aliases", {})
    candidates = [source_material, shape_name]
    for candidate in candidates:
        if not candidate:
            continue
        candidate_id = safe_name(str(candidate))
        if candidate_id in aliases:
            return aliases[candidate_id]
        if candidate_id in materials:
            return candidate_id
    searchable = " ".join(str(candidate) for candidate in candidates if candidate)
    for assignment in config.get("assignments", []):
        if re.search(assignment["pattern"], searchable, flags=re.IGNORECASE):
            return assignment["material"]
    return config["default_material"]

### 2.3 Generated PLY Mesh Files

`export_ascii_ply()` writes geometry-only ASCII PLY files. These files contain vertex coordinates and face indices only, with no `vertex_color` fields. They are generated under `generated_model/meshes/` and are human-readable, but Sionna/Mitsuba will warn that ASCII PLY is slower to parse.

In [36]:
def export_ascii_ply(mesh: Any, ply_path: Path) -> None:
    """Write geometry-only ASCII PLY so the generated meshes are human-readable."""
    vertices = np.asarray(mesh.vertices, dtype=np.float64)
    faces = np.asarray(mesh.faces, dtype=np.int64)
    with ply_path.open("w", encoding="ascii") as f:
        f.write("ply\n")
        f.write("format ascii 1.0\n")
        f.write(f"element vertex {len(vertices)}\n")
        f.write("property float x\nproperty float y\nproperty float z\n")
        f.write(f"element face {len(faces)}\n")
        f.write("property list uchar int vertex_indices\n")
        f.write("end_header\n")
        for x, y, z in vertices:
            f.write(f"{x:.9g} {y:.9g} {z:.9g}\n")
        for face in faces:
            f.write(f"{len(face)} {' '.join(str(int(index)) for index in face)}\n")

### 2.4 GLB Mesh Extraction and Axis Conversion

`scene_meshes()` extracts individual meshes from the GLB scene and keeps their visual material names when available. `transform_mesh_axes()` converts GLB Y-up coordinates into Sionna RT Z-up coordinates using `(x, y, z) -> (x, -z, y)`.

In [37]:
def scene_meshes(loaded: Any, trimesh: Any, fallback_name: str) -> list[tuple[str, Any, str | None]]:
    """Extract transformed meshes from a trimesh Scene while preserving material names."""
    if not isinstance(loaded, trimesh.Scene):
        return [(fallback_name, loaded, mesh_material_name(loaded))]
    meshes = []
    for node_name in loaded.graph.nodes_geometry:
        transform, geometry_name = loaded.graph[node_name]
        geometry = loaded.geometry.get(geometry_name)
        if geometry is None:
            continue
        mesh = geometry.copy()
        mesh.apply_transform(transform)
        if getattr(mesh, "is_empty", False):
            continue
        material_name = mesh_material_name(geometry) or mesh_material_name(mesh)
        mesh_name = safe_name(str(node_name))
        if material_name and not re.search(material_name, mesh_name, flags=re.IGNORECASE):
            mesh_name = material_name + "_" + mesh_name
        meshes.append((mesh_name, mesh, material_name))
    return meshes


def transform_mesh_axes(mesh: Any, source_up: str) -> Any:
    """Convert GLB Y-up coordinates to Sionna Z-up coordinates."""
    if source_up == "z":
        return mesh
    if source_up != "y":
        raise ValueError(f"Unsupported source up axis: {source_up}")
    transform = np.array([
        [1.0, 0.0, 0.0, 0.0],
        [0.0, 0.0, -1.0, 0.0],
        [0.0, 1.0, 0.0, 0.0],
        [0.0, 0.0, 0.0, 1.0],
    ], dtype=np.float64)
    mesh.apply_transform(transform)
    return mesh

### 2.5 Generated XML Scene File

`convert_glb_to_mitsuba_xml()` is the main conversion function. It writes `generated_model/output_precise.xml`, deletes any old XML and old `output_precise_*.ply` files for this model, writes fresh PLY meshes, and creates a Mitsuba XML scene that Sionna RT can load with `load_scene()`.

In [38]:
def convert_glb_to_mitsuba_xml(model_path: Path, args: Any) -> Path:
    """Convert GLB geometry to Sionna-readable Mitsuba XML and PLY files."""
    import trimesh

    config = validate_material_config(material_config())
    output_dir = Path(args.converted_dir).expanduser()
    if not output_dir.is_absolute():
        output_dir = model_path.parent / output_dir
    mesh_dir = output_dir / "meshes"
    mesh_dir.mkdir(parents=True, exist_ok=True)

    stem = safe_name(model_path.stem)
    xml_path = output_dir / f"{stem}.xml"
    if xml_path.exists():
        xml_path.unlink()
    for old_ply_path in mesh_dir.glob(f"{stem}_*.ply"):
        old_ply_path.unlink()

    loaded = trimesh.load(model_path, force="scene")
    meshes = scene_meshes(loaded, trimesh, stem)
    if not meshes:
        raise ValueError(f"No mesh geometry found in {model_path}")

    material_xml = [material_to_xml(material_id, material) for material_id, material in config["materials"].items()]
    shape_xml = []
    used_names = set()
    material_usage = Counter()
    bounds_min = np.full(3, np.inf, dtype=np.float64)
    bounds_max = np.full(3, -np.inf, dtype=np.float64)

    for idx, (mesh_name, mesh, source_material) in enumerate(meshes):
        mesh = transform_mesh_axes(mesh.copy(), args.source_up)
        unique_name = safe_name(mesh_name)
        if unique_name in used_names:
            unique_name = f"{unique_name}_{idx}"
        used_names.add(unique_name)

        ply_path = mesh_dir / f"{stem}_{unique_name}.ply"
        export_ascii_ply(mesh, ply_path)
        bounds_min = np.minimum(bounds_min, mesh.bounds[0])
        bounds_max = np.maximum(bounds_max, mesh.bounds[1])

        material_id = material_for_shape(unique_name, config, source_material)
        material_usage[material_id] += 1
        shape_xml.append(textwrap.dedent(f"""\
            <shape type="ply" id="mesh-{escape(unique_name, quote=True)}">
                <string name="filename" value="meshes/{escape(ply_path.name, quote=True)}"/>
                <boolean name="face_normals" value="true"/>
                <ref id="{escape(material_id, quote=True)}" name="bsdf"/>
            </shape>""").rstrip())

    xml = "<scene version=\"2.1.0\">\n\n"
    xml += "<!-- Built-in radio materials -->\n"
    xml += "\n\n".join(material_xml)
    xml += "\n\n<!-- Shapes -->\n"
    xml += "\n\n".join(shape_xml)
    xml += "\n</scene>\n"
    xml_path.write_text(xml, encoding="utf-8")

    print(f"Converted {model_path.name} -> {xml_path}")
    print("  Material config: built-in material table")
    print(f"  Mesh objects: {len(shape_xml)}")
    print("  Material usage:")
    for material_id, count in material_usage.most_common():
        print(f"    - {material_id}: {count}")
    print(f"  Source up-axis: {args.source_up.upper()} -> Sionna Z-up")
    print(f"  Converted bounds: min={bounds_min.tolist()} max={bounds_max.tolist()}")
    return xml_path

## 3. Define Sionna RT Scene and 3D Preview Functions

These functions use Sionna RT APIs after the XML scene exists. They load the XML scene, set frequency and bandwidth, configure antenna arrays, and add TX/RX nodes. The actual interactive 3D model display is done later with Sionna's `scene.preview()`.

In [39]:
def scene_path_for_model(model_path: Path, args: Any) -> Path:
    """Return an XML scene path, converting GLB/GLTF inputs when needed."""
    suffix = model_path.suffix.lower()
    if suffix == ".xml":
        return model_path
    if suffix in {".glb", ".gltf"}:
        return convert_glb_to_mitsuba_xml(model_path, args)
    raise ValueError(f"Unsupported model format {model_path.suffix!r}. Use .xml, .glb, or .gltf.")


def build_scene(args: Any):
    """Load the Sionna scene and add arrays, transmitter, and receiver."""
    model_path = Path(args.model).expanduser().resolve()
    if not model_path.exists():
        raise FileNotFoundError(f"Model file does not exist: {model_path}")

    scene = load_scene(str(scene_path_for_model(model_path, args)))
    scene.frequency = args.frequency
    scene.bandwidth = args.bandwidth
    scene.tx_array = PlanarArray(
        num_rows=args.tx_rows,
        num_cols=args.tx_cols,
        pattern=args.tx_pattern,
        polarization=args.tx_polarization,
    )
    scene.rx_array = PlanarArray(
        num_rows=args.rx_rows,
        num_cols=args.rx_cols,
        pattern=args.rx_pattern,
        polarization=args.rx_polarization,
    )
    scene.add(Transmitter(name="base_station", position=args.tx, look_at=args.look_at, power_dbm=args.tx_power_dbm))
    scene.add(Receiver(name="user_terminal", position=args.rx))
    return scene


def print_scene_summary(scene, args: Any) -> None:
    """Print a minimal scene summary without extra helper formatting."""
    print("Loaded scene")
    print(f"  Objects: {len(scene.objects)}")
    print(f"  Radio materials: {len(scene.radio_materials)}")
    print(f"  Frequency: {args.frequency / 1e9:.3f} GHz")
    print(f"  Bandwidth: {args.bandwidth / 1e6:.3f} MHz")

## 4. Convert GLB and Load the Sionna RT Scene

This cell runs both sides of the pipeline: first it converts the GLB to XML/PLY if needed, then it loads the XML with Sionna RT and adds TX/RX.

In [40]:
scene = build_scene(args)
print_scene_summary(scene, args)


Converted example_chongqing.glb -> /root/glb_to_rt/generated_model/example_chongqing.xml
  Material config: built-in material table
  Mesh objects: 6116
  Material usage:
    - asphalt: 3407
    - brick: 1812
    - concrete: 619
    - grass: 152
    - glass: 85
    - forest: 30
    - water: 9
    - sand: 1
    - soil: 1
  Source up-axis: Y -> Sionna Z-up
  Converted bounds: min=[-9320.48046875, -7392.97216796875, 0.0] max=[3786.333740234375, 5713.17236328125, 627.7716674804688]
2026-07-09 11:04:44 WARN  [PLYMesh] "example_chongqing_land_dem_precise_soil_fill.ply": performance warning -- this file uses the ASCII PLY format, which is slow to parse. Consider converting it to the binary PLY format.
Loaded scene
  Objects: 9
  Radio materials: 9
  Frequency: 3.500 GHz
  Bandwidth: 100.000 MHz


## 5. Interactive 3D Preview

This part is only for visualization. It calls Sionna RT's built-in `scene.preview()` on the already loaded scene.

In [41]:
scene.preview(
    resolution=tuple(args.preview_resolution),
    fov=args.preview_fov,
    show_devices=not args.hide_devices,
    show_orientations=args.show_orientations,
)


## 6. Optional Ray Tracing

This part is only for propagation computation. It uses Sionna RT's `PathSolver()` with the scene, TX, RX, arrays, and materials that were already configured above.

In [42]:
def shape_text(value: Any) -> str:
    """Return shapes for tensors or nested tuple/list outputs from Sionna RT."""
    if value is None:
        return "None"
    if isinstance(value, (tuple, list)):
        return "(" + ", ".join(shape_text(item) for item in value) + ")"
    shape = getattr(value, "shape", None)
    return str(shape) if shape is not None else type(value).__name__


path_solver = PathSolver()
paths = path_solver(scene, max_depth=args.max_depth, refraction=args.refraction)
print("Computed propagation paths")
for attr in ("a", "tau", "theta_t", "phi_t", "theta_r", "phi_r"):
    print(f"  {attr}: {shape_text(getattr(paths, attr, None))}")

Computed propagation paths
  a: ((1, 1, 1, 16, 1), (1, 1, 1, 16, 1))
  tau: (1, 1, 1)
  theta_t: (1, 1, 1)
  phi_t: (1, 1, 1)
  theta_r: (1, 1, 1)
  phi_r: (1, 1, 1)
